# Agent Runtime へのデプロイ

このノートブックでは、ADK で作成した AI エージェント（AdkApp オブジェクト）を Agent Runtime にデプロイする方法を学びます。

## 事前準備

**[ARD-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[ARD-02]**

インストールされたパッケージのバージョンを確認します。

In [1]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[ARD-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [2]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

変数 PROJECT_ID を設定しました。
PROJECT_ID = etsuji-15pro-poc


**[ARD-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

**[ARD-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [4]:
import os
from IPython.display import HTML, Markdown, display
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk.agents.llm_agent import LlmAgent
from google.adk.tools import google_search

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

## LlmAgent オブジェクトと AdkApp オブジェクトの作成

**[ARD-06]**

Grounding with Google Search を利用して、ユーザーの質問に回答する AI エージェント（LlmAgent オブジェクト）を作成します。

In [5]:
instruction = '''
あなたはユーザーの質問に回答するエージェントです。
- google_search を使用して、最新情報に基づいて回答してください。
- フレンドリーな会話を心がけてください。
'''

search_agent = LlmAgent(
    name='search_agent',
    model='gemini-3.5-flash-lite',
    description='Google 検索を用いて質問に回答するエージェント',
    instruction=instruction,
    tools=[google_search],
)

search_agent_app = AdkApp(
    agent=search_agent,
    app_name='search_agent_app',
)

## AdkApp オブジェクトと会話するアプリケーションの作成

**[ARD-08]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

このクラスは、ローカルの AdkApp と Agent Runtime にデプロイしたリモートの AdkApp のどちらでも利用できます。

In [6]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
        return '\n'.join(result)

## ローカルでの動作確認

**[ARD-09]**

ローカルの AdkApp オブジェクト `search_agent_app` を使用する ChatClient オブジェクトを作成して、動作確認をします。

In [20]:
chat_client = ChatClient(search_agent_app)

query = '''
今日の渋谷区の天気は？
'''
response = await chat_client.async_stream_query(query)
display(Markdown(response))

今日（9月1日）の渋谷区は、朝は雲が広がりますが、日中は日差しの出る時間もある見込みです。雨の心配は比較的少なく、お天気は穏やかですが、最高気温は31℃前後まで上がって昨日よりも暑さが増しそうです。

蒸し暑さ続きますので、通気性の良い服装を選んだり、こまめな水分補給を心がけたりして熱中症対策をなさってくださいね。お出かけの際は安心してお過ごしいただけますが、紫外線対策などもあわせてどうぞ！

## Agent Runtime へのデプロイ

**[ARD-10]**

Agent Runtime へのデプロイに必要な設定値 `config` を用意します。

`display_name` はデプロイしたリソースの表示名（人間が見てわかる名前）を指定します。デプロイが完了すると、これとは別にユニークなリソース名が割り当てられます。

In [17]:
display_name = 'Search Agent App'

requirements = [
    'google-adk==2.8.0',
    'google-cloud-aiplatform==2.0.1',
    'google-genai==2.20.0',
]

config={
    'agent_framework': 'google-adk',
    'requirements': requirements,
    'staging_bucket': f'gs://{PROJECT_ID}_search_agent_app',
    'display_name': display_name,
    'env_vars': {
        'GOOGLE_CLOUD_LOCATION': 'global',
        'GOOGLE_GENAI_USE_VERTEXAI': 'True',
    },
}

**[ARD-11]**

Agent Runtime を操作するクライアントオブジェクトを変数 `agent_runtime` に保存します。

In [ ]:
agent_runtime = agentplatform.Client(location='us-central1').runtimes

**[ARD-12]**

クライアントオブジェクトを利用して、変数 `search_agent_app` に保存された AdkApp オブジェクトを Agent Runtime にデプロイします。

In [19]:
remote_adk_app = agent_runtime.create(
    agent=search_agent_app,
    config=config,
)

INFO:agentplatform_genai.runtimes:View progress and logs at https://console.cloud.google.com/logs/query?project=agent-development-507109&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%221309392454599835648%22.
INFO:agentplatform_genai.runtimes:Agent Runtime created. To use it in another session:
INFO:agentplatform_genai.runtimes:runtime=client.runtimes.get(name='projects/848570887571/locations/us-central1/reasoningEngines/1309392454599835648')


デプロイ完了まで数分かかります。デプロイの進捗状況は、出力メッセージにある Cloud Logging へのリンクから確認できます。

```
INFO:agentplatform_genai.runtimes:View progress and logs at https://console.cloud.google.com/logs/query?project=...
```

デプロイ完了時の出力メッセージから、デプロイされた AdkApp オブジェクトのリソース名が確認できます。

```
INFO:agentplatform_genai.runtimes:runtime=client.runtimes.get(name='projects/848570887571/locations/us-central1/reasoningEngines/1309392454599835648')
```

## Agent Runtime 上のリソースの利用

**[ARD-13]**

デプロイ済みのリソースを一覧表示します。

In [17]:
for item in agent_runtime.list():
    resource = item.api_resource
    print(f'{resource.display_name}: {resource.name}')

Search Agent App: projects/848570887571/locations/us-central1/reasoningEngines/1309392454599835648


なお、**[ARD-12]** で実行したデプロイコマンドが完了すると、変数 `remote_adk_app` には、デプロイした AdkApp オブジェクトを操作するためのクライアントオブジェクトが格納されています。クライアントオブジェクトを再取得する際は、確認したリソース名を指定して、次のコマンドを実行します。

```
agent_runtime = agentplatform.Client(location='us-central1').runtimes
remote_adk_app = agent_runtime.get(name='projects/848570887571/locations/us-central1/reasoningEngines/1309392454599835648')
```

**[ARD-14]**

変数 `remote_adk_app` のクライアントオブジェクトは、ローカルの AdkApp オブジェクトと同じメソッド（`async_create_session`, `async_stream_query` など）を持つので、ChatClient オブジェクトのアプリから AdkApp オブジェクトと同様に利用できます。

In [19]:
chat_client = ChatClient(remote_adk_app)

query = '''
今日の新宿区の天気は？
'''
response = await chat_client.async_stream_query(query)
display(Markdown(response))

今日（9月1日）の新宿区は、朝は雲が広がっていますが、昼前からは日差しが届く見込みです。夕方以降は再び雲が多くなりますが、雨の心配はほとんどありません。

* **天気**：曇りのち晴れ（日中はおおむね晴れ間が出ます）
* **最高気温**：31℃前後
* **服装**：真夏のような暑さや蒸し暑さが続くため、半袖など風通しのよい服装がおすすめです。熱中症対策や、室内外の温度差への備えを意識してお過ごしくださいね。

## （参考）Agent Runtime 上のリソースの管理

デプロイ済みのリソースを更新する際は、**[ARD-10]** で用意した設定を用いて、次のコマンドを実行します。`resource_name` に更新対象のリソース名を指定します。

```
agent_runtime = agentplatform.Client(location='us-central1').runtimes
resource_name = 'projects/848570887571/locations/us-central1/reasoningEngines/1309392454599835648'
remote_adk_app = agent_runtime.update(
    name=resource_name,
    agent=search_agent_app,
    config=config,
)
```

デプロイしたリソースを削除する際は、削除対象のリソースを操作するクライアントオブジェクトを取得して、`delete` メソッドを実行します。

```
agent_runtime = agentplatform.Client(location='us-central1').runtimes
remote_adk_app = agent_runtime.get(name='projects/848570887571/locations/us-central1/reasoningEngines/1309392454599835648')
remote_adk_app.delete(force=True)
```

## ソースファイルからデプロイする方法

AdkApp オブジェクトを定義するソースファイルからデプロイする方法を説明します。

**[ARD-15]**

ディレクトリ `search_agent_app` の下に `agent.py` というファイル名で、AdkApp オブジェクトを定義するソースファイルを作成します。

In [28]:
%%bash
mkdir -p search_agent_app
cat <<'EOF' >search_agent_app/agent.py
import os
from agentplatform.frameworks import AdkApp
from google.adk.agents import LlmAgent
from google.adk.tools import google_search

instruction = '''
あなたはユーザーの質問に回答するエージェントです。
- google_search を使用して、最新情報に基づいて回答してください。
- フレンドリーな会話を心がけてください。
'''

search_agent = LlmAgent(
    name='search_agent',
    model='gemini-3.5-flash-lite',
    description='Google 検索を用いて質問に回答するエージェント',
    instruction=instruction,
    tools=[google_search],
)

search_agent_app = AdkApp(
    agent=search_agent,
    app_name=os.environ.get('GOOGLE_CLOUD_AGENT_ENGINE_ID', 'default-app-name'),
)
EOF

**[ARD-16]**

同じディレクトリに、デプロイ用のコンテナイメージに追加でインストールするパッケージを指定した `requirements.txt` を作成します。

In [ ]:
%%bash
cat <<'EOF' >search_agent_app/requirements.txt
google-adk==2.8.0
google-cloud-aiplatform==2.0.1
google-genai==2.20.0
EOF

**[ARD-17]**

次のコマンドで、ディレクトリ `search_agent_app` 以下のソースファイルからデプロイ用のコンテナイメージを作成して、Agent Runtime にデプロイします。

In [29]:
agent_runtime = agentplatform.Client(location='us-central1').runtimes

display_name = 'Search Agent App2'

config = {
    'source_packages': ['search_agent_app'],
    'entrypoint_module': 'search_agent_app.agent',  # ファイル agent.py に対応するモジュール名
    'entrypoint_object': 'search_agent_app',        # AdkApp オブジェクトを格納した変数名
    'requirements_file': 'search_agent_app/requirements.txt',
    'class_methods': [
        {
            'name': 'async_stream_query',
            'api_mode': 'async_stream',
            'description': 'Stream responses from the agent for a given query.',
        },
        {
            'name': 'async_create_session',
            'api_mode': 'async',
            'description': 'Create a new managed session.',
        },
    ],
    'display_name': display_name,
    'env_vars': {
        'GOOGLE_CLOUD_LOCATION': 'global',
        'GOOGLE_GENAI_USE_VERTEXAI': 'True',
    },
}

remote_agent = agent_runtime.create(
    config=config,
)

INFO:agentplatform_genai.runtimes:View progress and logs at https://console.cloud.google.com/logs/query?project=agent-development-507109&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%225825376980945600512%22.
INFO:agentplatform_genai.runtimes:Agent Runtime created. To use it in another session:
INFO:agentplatform_genai.runtimes:runtime=client.runtimes.get(name='projects/848570887571/locations/us-central1/reasoningEngines/5825376980945600512')
